In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Simple scalar encoder, decoder, predictor
class Encoder(nn.Module):
    def __init__(self, w_init):
        super().__init__()
        self.w = nn.Parameter(torch.tensor(w_init))

    def forward(self, x):
        return self.w * x

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Linear(1, 1)

    def forward(self, z):
        return self.w(z)

class Predictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Linear(1, 1)

    def forward(self, z):
        return self.w(z)

def train_model(w_init, method='projected', epochs=500, lambda_pred=0.1):
    encoder = Encoder(w_init)
    decoder = Decoder()
    predictor = Predictor()
    opt = optim.SGD(list(encoder.parameters()) + list(decoder.parameters()) + list(predictor.parameters()), lr=0.1)

    for epoch in range(epochs):
        t = torch.rand((20,1))
        o = torch.sin(t)
        o_next = torch.sin(t + 1)

        opt.zero_grad()
        z = encoder(o)
        recon = decoder(z)
        pred = predictor(z)

        loss_recon = ((recon - o) ** 2).mean()
        loss_pred = ((pred - encoder(o_next).detach()) ** 2).mean()

        if method == 'projected':
            grad_recon = torch.autograd.grad(loss_recon, z, create_graph=True)[0]
            grad_pred = torch.autograd.grad(loss_pred, z, create_graph=True)[0]

            dot = (grad_recon * grad_pred).sum()
            proj = dot / (grad_pred.norm() ** 2 + 1e-8) * grad_pred
            grad_recon_proj = grad_recon - proj

            grad_combined = grad_pred + lambda_pred * grad_recon_proj
            z.backward(grad_combined)

            loss = loss_recon + loss_pred
            grads = torch.autograd.grad(loss, list(decoder.parameters()) + list(predictor.parameters()))
            for p, g in zip(list(decoder.parameters()) + list(predictor.parameters()), grads):
                p.grad = g
        else:
            loss = loss_recon + lambda_pred * loss_pred
            loss.backward()

        opt.step()

    # Final evaluation
    with torch.no_grad():
        z = encoder(o)
        recon = decoder(z)
        pred = predictor(z)
        final_loss_recon = ((recon - o) ** 2).mean().item()
        final_loss_pred = ((pred - encoder(o_next)) ** 2).mean().item()
    return final_loss_recon, final_loss_pred

# Try different initial encoder weights
import tqdm
w_inits = torch.linspace(-2.0, 2.0, steps=100)
errors_proj = []
errors_sum = []
N = 5
for w in tqdm.tqdm(w_inits):
    recon_p, pred_p = 0.0, 0.0
    recon_s, pred_s = 0.0, 0.0
    for _ in range(N):
        r, p = train_model(w.item(), method='projected')
        recon_p += r
        pred_p += p

        r, p = train_model(w.item(), method='sum')
        recon_s += r
        pred_s += p

    errors_proj.append((recon_p / N, pred_p / N))
    errors_sum.append((recon_s / N, pred_s / N))
errors_proj = torch.tensor(errors_proj)
errors_sum = torch.tensor(errors_sum)

w_inits = w_inits.numpy()
proj_recon, proj_pred = errors_proj[:,0].numpy(), errors_proj[:,1].numpy()
sum_recon, sum_pred = errors_sum[:,0].numpy(), errors_sum[:,1].numpy()

plt.figure(figsize=(10, 5))
plt.plot(w_inits, proj_recon, label='Recon (projected)', color='blue')
plt.plot(w_inits, proj_pred, label='Pred (projected)', color='cyan')
plt.plot(w_inits, sum_recon, label='Recon (sum)', color='red')
plt.plot(w_inits, sum_pred, label='Pred (sum)', color='orange')
plt.xlabel("Initial encoder weight")
plt.ylabel("Final loss")
plt.title("Final reconstruction and prediction loss vs. encoder init")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


 47%|████▋     | 47/100 [01:21<01:31,  1.73s/it]


KeyboardInterrupt: 